In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
from geopy.geocoders import Nominatim

In [2]:
STATCAN_DIR = Path("data/external/statscan")

CANDIDATE_PATH = (
    STATCAN_DIR /
    "filtered/gta_census_demographic_candidates.parquet"
)

TRACT_MAP_PATH = (
    STATCAN_DIR /
    "gta_store_census_tract_mapping.parquet"
)

OUTPUT_PATH = (
    STATCAN_DIR /
    "gta_store_demographics_v1.parquet"
)

# Load GTA Walmart stores

In [3]:
candidates = pd.read_parquet(CANDIDATE_PATH)
tract_map = pd.read_parquet(TRACT_MAP_PATH)

print("Candidates:", candidates.shape)
print("Tract map:", tract_map.shape)

display(candidates.head())
display(tract_map.head())

Candidates: (146410, 5)
Tract map: (1103, 6)


,DGUID,GEO_NAME,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL
0,2021S05075350031.00,5350031.00,1402,Total - Indigenous identity for the population...,5470
1,2021S05075350031.00,5350031.00,1411,Total - Registered or Treaty Indian status for...,5470
2,2021S05075350031.00,5350031.00,1414,Total - Private households by tenure - 25% sam...,2670
3,2021S05075350031.00,5350031.00,1416,Renter,2160
4,2021S05075350031.00,5350031.00,1434,Total - Private households by number of person...,2670


,store_id,store_name,DGUID,intersection_area_m2,tract_area_m2,area_weight
0,3106,Toronto Dufferin Mall,2021S05075350010.01,247461.063726,2.474611e+05,1.000000
1,3106,Toronto Dufferin Mall,2021S05075350010.02,537964.815388,5.379648e+05,1.000000
2,3106,Toronto Dufferin Mall,2021S05075350002.00,992806.909061,3.395314e+06,0.292405
3,3106,Toronto Dufferin Mall,2021S05075350003.00,958512.541558,9.585125e+05,1.000000
4,3106,Toronto Dufferin Mall,2021S05075350004.00,340251.728336,3.402517e+05,1.000000


In [4]:
candidates["DGUID"] = candidates["DGUID"].astype(str)

candidates["CHARACTERISTIC_ID"] = (
    candidates["CHARACTERISTIC_ID"]
    .astype(str)
    .str.strip()
)

candidates["C1_COUNT_TOTAL"] = pd.to_numeric(
    candidates["C1_COUNT_TOTAL"],
    errors="coerce"
)

tract_map["DGUID"] = tract_map["DGUID"].astype(str)

tract_map["area_weight"] = pd.to_numeric(
    tract_map["area_weight"],
    errors="coerce"
)

In [5]:
def find_var(pattern, n=30):
    x = (
        candidates[
            candidates["CHARACTERISTIC_NAME"]
            .astype(str)
            .str.contains(
                pattern,
                case=False,
                na=False,
                regex=True
            )
        ][
            [
                "CHARACTERISTIC_ID",
                "CHARACTERISTIC_NAME"
            ]
        ]
        .drop_duplicates()
    )

    return x.head(n)

In [6]:
display(
    find_var(
        r"^Population, 2021$"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
72,1,"Population, 2021"


In [7]:
display(
    find_var(
        r"population density"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
75,6,Population density per square kilometre


In [8]:
display(
    find_var(
        r"average.*household.*size"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
82,57,Average household size


In [9]:
display(
    find_var(
        r"^Employment rate"
    )
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
60,2229,Employment rate


In [10]:
display(
    candidates[
        candidates["CHARACTERISTIC_ID"] == "243"
    ][
        [
            "CHARACTERISTIC_ID",
            "CHARACTERISTIC_NAME"
        ]
    ]
    .drop_duplicates()
)

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
174,243,Median total income of household in 2020 ($)


# Automatically identify the main variables

In [11]:
def get_first_id(pattern):
    result = (
        candidates[
            candidates["CHARACTERISTIC_NAME"]
            .astype(str)
            .str.contains(
                pattern,
                case=False,
                na=False,
                regex=True
            )
        ][
            [
                "CHARACTERISTIC_ID",
                "CHARACTERISTIC_NAME"
            ]
        ]
        .drop_duplicates()
    )

    if len(result) == 0:
        return None

    return result.iloc[0]["CHARACTERISTIC_ID"]

In [12]:
VARIABLE_IDS = {
    "population": get_first_id(
        r"^Population, 2021$"
    ),

    "population_density": get_first_id(
        r"population density"
    ),

    "average_household_size": get_first_id(
        r"average.*household.*size"
    ),

    "employment_rate": get_first_id(
        r"^Employment rate"
    ),

    "median_household_income": "243"
}

VARIABLE_IDS

{'population': '1',
 'population_density': '6',
 'average_household_size': '57',
 'employment_rate': '2229',
 'median_household_income': '243'}

# Verify Them

In [13]:
for feature, characteristic_id in VARIABLE_IDS.items():

    print("\n", feature, "→", characteristic_id)

    display(
        candidates[
            candidates["CHARACTERISTIC_ID"]
            == str(characteristic_id)
        ][
            [
                "CHARACTERISTIC_ID",
                "CHARACTERISTIC_NAME"
            ]
        ]
        .drop_duplicates()
    )


 population → 1


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
72,1,"Population, 2021"



 population_density → 6


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
75,6,Population density per square kilometre



 average_household_size → 57


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
82,57,Average household size



 employment_rate → 2229


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
60,2229,Employment rate



 median_household_income → 243


,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
174,243,Median total income of household in 2020 ($)


# Build tract-level feature table

In [15]:
selected_ids = {
    str(v): k
    for k, v in VARIABLE_IDS.items()
    if v is not None
}

selected = candidates[
    candidates["CHARACTERISTIC_ID"]
    .isin(selected_ids.keys())
].copy()

selected["feature"] = (
    selected["CHARACTERISTIC_ID"]
    .map(selected_ids)
)

print(selected.shape)

display(
    selected[
        [
            "DGUID",
            "feature",
            "C1_COUNT_TOTAL"
        ]
    ].head()
)

(3025, 6)


,DGUID,feature,C1_COUNT_TOTAL
60,2021S05075350031.00,employment_rate,48.8
72,2021S05075350032.00,population,6042.0
75,2021S05075350032.00,population_density,14396.0
82,2021S05075350032.00,average_household_size,1.6
174,2021S05075350032.00,median_household_income,41600.0


In [16]:
dup_check = (
    selected
    .groupby(
        ["DGUID", "feature"]
    )
    .size()
    .reset_index(name="n")
)

display(
    dup_check[
        dup_check["n"] > 1
    ].head(20)
)

,DGUID,feature,n


# Pivot to one row per Census Tract

In [17]:
tract_features = (
    selected
    .pivot_table(
        index="DGUID",
        columns="feature",
        values="C1_COUNT_TOTAL",
        aggfunc="first"
    )
    .reset_index()
)

tract_features.columns.name = None

print(tract_features.shape)

display(tract_features.head())

(605, 6)


,DGUID,average_household_size,employment_rate,median_household_income,population,population_density
0,2021S05075350001.00,2.6,78.1,137000.0,599.0,87.8
1,2021S05075350002.00,2.1,43.8,85000.0,604.0,178.0
2,2021S05075350003.00,1.7,72.5,89000.0,457.0,483.3
3,2021S05075350004.00,1.8,54.3,50000.0,6306.0,18525.3
4,2021S05075350005.00,1.8,58.5,66000.0,6957.0,18483.0


 # Join Census features to Walmart catchments

In [18]:
store_tracts = tract_map.merge(
    tract_features,
    on="DGUID",
    how="left"
)

print(store_tracts.shape)

display(store_tracts.head())

(1103, 11)


,store_id,store_name,DGUID,intersection_area_m2,tract_area_m2,area_weight,average_household_size,employment_rate,median_household_income,population,population_density
0,3106,Toronto Dufferin Mall,2021S05075350010.01,247461.063726,2.474611e+05,1.000000,1.7,77.6,110000.0,5611.0,22661.6
1,3106,Toronto Dufferin Mall,2021S05075350010.02,537964.815388,5.379648e+05,1.000000,1.7,75.4,97000.0,7468.0,13875.9
2,3106,Toronto Dufferin Mall,2021S05075350002.00,992806.909061,3.395314e+06,0.292405,2.1,43.8,85000.0,604.0,178.0
3,3106,Toronto Dufferin Mall,2021S05075350003.00,958512.541558,9.585125e+05,1.000000,1.7,72.5,89000.0,457.0,483.3
4,3106,Toronto Dufferin Mall,2021S05075350004.00,340251.728336,3.402517e+05,1.000000,1.8,54.3,50000.0,6306.0,18525.3


# Missingness

In [19]:
feature_cols = list(VARIABLE_IDS.keys())

missing = (
    store_tracts[feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

display(missing)

average_household_size     0.003626
employment_rate            0.003626
median_household_income    0.003626
population                 0.000000
population_density         0.000000
dtype: float64

# Weighted aggregation

In [20]:
store_tracts["weighted_population"] = (
    store_tracts["population"]
    * store_tracts["area_weight"]
)

# Weighted mean helper

In [21]:
def weighted_mean(values, weights):

    mask = (
        values.notna()
        & weights.notna()
        & (weights > 0)
    )

    if mask.sum() == 0:
        return np.nan

    return np.average(
        values[mask],
        weights=weights[mask]
    )

# Aggregate per store

In [22]:
rows = []

for store_id, group in store_tracts.groupby("store_id"):

    row = {
        "store_id": store_id,

        "store_name": (
            group["store_name"].iloc[0]
            if "store_name" in group.columns
            else None
        ),

        "population_5km": (
            group["weighted_population"]
            .sum()
        ),

        "population_density_5km": weighted_mean(
            group["population_density"],
            group["weighted_population"]
        ),

        "median_household_income_5km": weighted_mean(
            group["median_household_income"],
            group["weighted_population"]
        ),

        "average_household_size_5km": weighted_mean(
            group["average_household_size"],
            group["weighted_population"]
        ),

        "employment_rate_5km": weighted_mean(
            group["employment_rate"],
            group["weighted_population"]
        ),

        "n_census_tracts": (
            group["DGUID"].nunique()
        )
    }

    rows.append(row)

store_demographics = pd.DataFrame(rows)

# Inspect

In [23]:
print(store_demographics.shape)

display(
    store_demographics.sort_values(
        "population_5km",
        ascending=False
    )
)

(15, 8)


,store_id,store_name,population_5km,population_density_5km,median_household_income_5km,average_household_size_5km,employment_rate_5km,n_census_tracts
11,3106,Toronto Dufferin Mall,593613.628819,15197.462826,86252.286112,2.017781,62.389316,147
5,1150,Toronto Downtown East,482564.923619,16864.257792,88944.349902,2.079208,59.227363,118
0,1004,Toronto Stockyards,423374.384031,8590.969933,92072.905741,2.383569,56.936454,123
10,3105,Toronto North Park,373912.266144,6242.331511,85562.143106,2.569160,53.592422,100
12,3130,Brampton North,302827.194504,4676.046059,111511.199431,3.467075,58.465889,75
9,3055,Mississauga Square One,300634.761139,7762.848424,97477.248463,2.840229,53.343838,78
4,1139,Toronto Downsview,237963.886322,5979.220710,80848.194959,2.764884,51.253500,63
3,1126,Mississauga East,236119.747234,5766.186614,92757.843631,2.574732,54.538310,64
7,3031,Etobicoke North Queen,216141.913635,5515.568802,94646.324907,2.491546,55.562432,60
1,1061,Mississauga Heartland,213999.583891,6702.357845,114368.849113,3.166474,55.698902,58


# Sanity checks

In [24]:
store_demographics.describe(
    include="all"
).T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
store_id,15,15,1004,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_name,15,15,Toronto Stockyards,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
population_5km,15.0,NaN,NaN,NaN,276155.164225,138544.475096,98356.071055,181445.170179,236119.747234,338369.730324,593613.628819
population_density_5km,15.0,NaN,NaN,NaN,6979.903804,3926.8811,3686.40674,4478.306236,5766.186614,7232.603135,16864.257792
median_household_income_5km,15.0,NaN,NaN,NaN,99958.997165,14220.187736,80848.194959,90508.627822,95202.942801,109515.409362,128588.334413
average_household_size_5km,15.0,NaN,NaN,NaN,2.842754,0.529591,2.017781,2.530353,2.764884,3.216437,3.769682
employment_rate_5km,15.0,NaN,NaN,NaN,56.549045,3.059307,51.2535,54.902254,55.698902,58.846626,62.389316
n_census_tracts,15.0,NaN,NaN,NaN,73.533333,33.987112,27.0,52.5,63.0,89.0,147.0


In [25]:
print(
    "Stores:",
    store_demographics["store_id"].nunique()
)

print(
    "Missing values:"
)

display(
    store_demographics
    .isna()
    .sum()
)

Stores: 15
Missing values:


store_id                       0
store_name                     0
population_5km                 0
population_density_5km         0
median_household_income_5km    0
average_household_size_5km     0
employment_rate_5km            0
n_census_tracts                0
dtype: int64

# Save

In [26]:
store_demographics.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print(
    round(
        OUTPUT_PATH.stat().st_size / 1024**2,
        3
    ),
    "MB"
)

Saved: data/external/statscan/gta_store_demographics_v1.parquet
0.006 MB
